In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print(" Libraries imported successfully!")

 Libraries imported successfully!


In [3]:
df = pd.read_csv('European_Bank.csv')

In [4]:
print(f"Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset Loaded: 10000 rows, 14 columns


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
# 1. Clean the data: Drop columns we don't need
df_clean = df.drop(columns=['Surname'])

In [10]:
# 2. Create Age Groups (<30, 30-45, 46-60, 60+)
age_bins = [0, 29, 45, 60, np.inf]
age_labels = ['<30', '30–45', '46–60', '60+']
df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=age_bins, labels=age_labels)

In [11]:
# 3. Create Credit Score Bands (Low, Medium, High)
credit_bins = [0, 599, 750, np.inf]
credit_labels = ['Low (<600)', 'Medium (600-750)', 'High (>750)']
df_clean['CreditScoreBand'] = pd.cut(df_clean['CreditScore'], bins=credit_bins, labels=credit_labels)

In [17]:
# 4. Tenure Groups (New, Mid-term, Long-term customers)
df_clean['TenureGroup'] = pd.cut(
    df_clean['Tenure'],
    bins=[-1, 2, 7, np.inf],
    labels=['New', 'Mid-term', 'Long-term']
)

In [18]:
# 5. Create Balance Segments (Zero balance, Low balance, High balance)
balance_bins = [-1, 0, 100000, np.inf]
balance_labels = ['Zero-balance', 'Low-balance (<100k)', 'High-balance (100k+)']
df_clean['BalanceSegment'] = pd.cut(df_clean['Balance'], bins=balance_bins, labels=balance_labels)

In [19]:
# 6. Geographic Segmentation (Already in the dataset as 'Geography')
# Values: France, Spain, Germany

In [25]:
# Let's check our new columns!
print("Cleaned data with new segments preview:")
display(df_clean[['Geography', 'AgeGroup', 'CreditScoreBand', 'TenureGroup', 'BalanceSegment', 'Exited']].head())

Cleaned data with new segments preview:


,Geography,AgeGroup,CreditScoreBand,TenureGroup,BalanceSegment,Exited
0,France,30–45,Medium (600-750),New,Zero-balance,1
1,Spain,30–45,Medium (600-750),New,Low-balance (<100k),0
2,France,30–45,Low (<600),Long-term,High-balance (100k+),1
3,France,30–45,Medium (600-750),New,Zero-balance,0
4,Spain,30–45,High (>750),New,High-balance (100k+),0


In [21]:
import pandas as pd

# 1. Overall Churn Rate
total_customers = len(df_clean)
total_churned = df_clean['Exited'].sum()
overall_churn_rate = (total_churned / total_customers) * 100

print("="*50)
print(f"1. OVERALL CHURN RATE SUMMARY")
print("="*50)
print(f"Total Customers: {total_customers:,}")
print(f"Total Churned: {total_churned:,}")
print(f"Overall Churn Rate: {overall_churn_rate:.2f}%\n")

# 2. Segment-Wise Churn Rates & Churn Contribution
segments = ['Geography', 'AgeGroup', 'CreditScoreBand', 'TenureGroup', 'BalanceSegment']

for seg in segments:
    print("="*50)
    print(f"Segment Analysis: {seg}")
    print("="*50)

    # Calculate count, churned count, and churn rate %
    seg_summary = df_clean.groupby(seg, observed=False)['Exited'].agg(
        Total_Customers='count',
        Churned_Customers='sum'
    ).reset_index()

    seg_summary['Churn Rate (%)'] = (seg_summary['Churned_Customers'] / seg_summary['Total_Customers']) * 100

    # Calculate Churn Contribution (% of total churned customers)
    seg_summary['Churn Contribution (%)'] = (seg_summary['Churned_Customers'] / total_churned) * 100

    display(seg_summary.round(2))
    print("\n")

# 3. Comparison of Churned vs Retained Profiles (Averages)
print("="*50)
print("3. PROFILE COMPARISON: RETAINED (0) VS CHURNED (1)")
print("="*50)

profile_comparison = df_clean.groupby('Exited')[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']].mean().reset_index()
profile_comparison['Exited'] = profile_comparison['Exited'].map({0: 'Retained (0)', 1: 'Churned (1)'})
display(profile_comparison.round(2))

1. OVERALL CHURN RATE SUMMARY
Total Customers: 10,000
Total Churned: 2,037
Overall Churn Rate: 20.37%

Segment Analysis: Geography


,Geography,Total_Customers,Churned_Customers,Churn Rate (%),Churn Contribution (%)
0,France,5014,810,16.15,39.76
1,Germany,2509,814,32.44,39.96
2,Spain,2477,413,16.67,20.27




Segment Analysis: AgeGroup


,AgeGroup,Total_Customers,Churned_Customers,Churn Rate (%),Churn Contribution (%)
0,<30,1641,124,7.56,6.09
1,30–45,6248,956,15.30,46.93
2,46–60,1647,842,51.12,41.34
3,60+,464,115,24.78,5.65




Segment Analysis: CreditScoreBand


...



Segment Analysis: BalanceSegment


,BalanceSegment,Total_Customers,Churned_Customers,Churn Rate (%),Churn Contribution (%)
0,Zero-balance,3617,500,13.82,24.55
1,Low-balance (<100k),1584,326,20.58,16.00
2,High-balance (100k+),4799,1211,25.23,59.45




3. PROFILE COMPARISON: RETAINED (0) VS CHURNED (1)


,Exited,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary
0,Retained (0),651.85,37.41,5.03,72745.30,1.54,99738.39
1,Churned (1),645.35,44.84,4.93,91108.54,1.48,101465.68


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [22]:
import pandas as pd

print("="*50)
print("1. GENDER-BASED CHURN DIFFERENCES")
print("="*50)

gender_churn = df_clean.groupby('Gender', observed=False)['Exited'].agg(
    Total_Customers='count',
    Churned_Customers='sum'
).reset_index()

gender_churn['Churn Rate (%)'] = (gender_churn['Churned_Customers'] / gender_churn['Total_Customers']) * 100
display(gender_churn.round(2))


print("\n" + "="*50)
print("2. GEOGRAPHY-AGE INTERACTION ANALYSIS (CHURN RATE %)")
print("="*50)

# Creating a pivot table to see churn rates across Country + Age Group
geo_age_rate = df_clean.pivot_table(
    index='Geography',
    columns='AgeGroup',
    values='Exited',
    aggfunc='mean',
    observed=False
) * 100

display(geo_age_rate.round(2))


print("\n" + "="*50)
print("3. FINANCIAL STABILITY VS CHURN (CREDIT SCORE + BALANCE SEGMENTS)")
print("="*50)

fin_stability = df_clean.groupby(['CreditScoreBand', 'BalanceSegment'], observed=False)['Exited'].agg(
    Total_Customers='count',
    Churned_Customers='sum'
).reset_index()

fin_stability['Churn Rate (%)'] = (fin_stability['Churned_Customers'] / fin_stability['Total_Customers']) * 100

# Sort by highest churn rate to see most vulnerable financial profiles
fin_stability_sorted = fin_stability[fin_stability['Total_Customers'] > 5].sort_values(by='Churn Rate (%)', ascending=False)
display(fin_stability_sorted.round(2))

1. GENDER-BASED CHURN DIFFERENCES


,Gender,Total_Customers,Churned_Customers,Churn Rate (%)
0,Female,4543,1139,25.07
1,Male,5457,898,16.46



2. GEOGRAPHY-AGE INTERACTION ANALYSIS (CHURN RATE %)


AgeGroup,<30,30–45,46–60,60+
Geography,,,,
France,5.20,11.87,45.79,19.91
Germany,12.63,25.30,67.33,38.94
Spain,7.94,12.52,40.66,20.83



3. FINANCIAL STABILITY VS CHURN (CREDIT SCORE + BALANCE SEGMENTS)


,CreditScoreBand,BalanceSegment,Total_Customers,Churned_Customers,Churn Rate (%)
2,Low (<600),High-balance (100k+),1452,391,26.93
5,Medium (600-750),High-balance (100k+),2551,630,24.70
8,High (>750),High-balance (100k+),796,190,23.87
1,Low (<600),Low-balance (<100k),458,106,23.14
7,High (>750),Low-balance (<100k),250,51,20.40
4,Medium (600-750),Low-balance (<100k),876,169,19.29
0,Low (<600),Zero-balance,1124,163,14.50
3,Medium (600-750),Zero-balance,1941,265,13.65
6,High (>750),Zero-balance,552,72,13.04


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [23]:
import pandas as pd

print("="*50)
print("1. IDENTIFY HIGH-BALANCE CHURNERS")
print("="*50)

# Filter for customers who churned AND are in the High-balance tier
high_bal_churners = df_clean[(df_clean['Exited'] == 1) & (df_clean['BalanceSegment'] == 'High-balance')]
print(f"Total High-Balance Customers Who Churned: {len(high_bal_churners):,}")
display(high_bal_churners[['CustomerId', 'Geography', 'Age', 'CreditScore', 'Balance', 'EstimatedSalary']].head())


print("\n" + "="*50)
print("2. SALARY VS BALANCE CHURN PATTERNS")
print("="*50)

# Create Salary Bands to compare with Balance Segments
df_clean['SalaryBand'] = pd.qcut(df_clean['EstimatedSalary'], q=3, labels=['Low Salary', 'Medium Salary', 'High Salary'])

# Pivot table: Churn rate by Salary Band and Balance Segment
salary_balance_churn = df_clean.pivot_table(
    index='SalaryBand',
    columns='BalanceSegment',
    values='Exited',
    aggfunc='mean',
    observed=False
) * 100

print("Churn Rate (%) Matrix by Salary Band & Balance Segment:")
display(salary_balance_churn.round(2))


print("\n" + "="*50)
print("3. QUANTIFYING REVENUE / CAPITAL RISK FROM CHURN")
print("="*50)

# Total balance lost across all churned customers
total_balance_lost = df_clean[df_clean['Exited'] == 1]['Balance'].sum()
high_balance_lost = high_bal_churners['Balance'].sum()

print(f"Total Account Balance Lost (All Churned Customers): €{total_balance_lost:,.2f}")
print(f"Capital Lost from High-Balance Churners (€100k+): €{high_balance_lost:,.2f}")

# Percentage of capital risk concentrated in high-balance churners
capital_risk_pct = (high_balance_lost / total_balance_lost) * 100
print(f"Share of Capital Risk from High-Balance Segment: {capital_risk_pct:.1f}%")

1. IDENTIFY HIGH-BALANCE CHURNERS
Total High-Balance Customers Who Churned: 0


,CustomerId,Geography,Age,CreditScore,Balance,EstimatedSalary



2. SALARY VS BALANCE CHURN PATTERNS
Churn Rate (%) Matrix by Salary Band & Balance Segment:


BalanceSegment,Zero-balance,Low-balance (<100k),High-balance (100k+)
SalaryBand,,,
Low Salary,13.12,16.67,26.67
Medium Salary,14.04,22.32,23.24
High Salary,14.33,22.62,25.79



3. QUANTIFYING REVENUE / CAPITAL RISK FROM CHURN
Total Account Balance Lost (All Churned Customers): €185,588,094.63
Capital Lost from High-Balance Churners (€100k+): €0.00
Share of Capital Risk from High-Balance Segment: 0.0%
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [24]:
import pandas as pd

print("="*60)
print("OFFICIAL PROJECT KEY PERFORMANCE INDICATORS (KPIs) SUMMARY")
print("="*60)

# 1. Overall Churn Rate
total_customers = len(df_clean)
total_churned = df_clean['Exited'].sum()
overall_churn_rate = (total_churned / total_customers) * 100

# 2. Segment Churn Rates (Max and Min across segments for summary)
age_churn = df_clean.groupby('AgeGroup', observed=False)['Exited'].mean() * 100
geo_churn = df_clean.groupby('Geography', observed=False)['Exited'].mean() * 100

# 3. High-Value Churn Ratio (Churn rate among high-balance premium customers, e.g., Balance > 100k)
high_val_customers = df_clean[df_clean['BalanceSegment'] == 'High-balance']
high_val_churn_ratio = (high_val_customers['Exited'].sum() / len(high_val_customers)) * 100 if len(high_val_customers) > 0 else 0

# 4. Geographic Risk Index (Ratio of highest regional churn rate vs lowest)
max_geo_churn = geo_churn.max()
min_geo_churn = geo_churn.min()
geo_risk_index = max_geo_churn / min_geo_churn if min_geo_churn > 0 else 0

# 5. Engagement Drop Indicator (Difference in churn rate between inactive vs active members)
inactive_churn_rate = df_clean[df_clean['IsActiveMember'] == 0]['Exited'].mean() * 100
active_churn_rate = df_clean[df_clean['IsActiveMember'] == 1]['Exited'].mean() * 100
engagement_drop_indicator = inactive_churn_rate - active_churn_rate

# Create a clean DataFrame for the KPIs
kpi_summary = pd.DataFrame({
    'KPI Name': [
        'Overall Churn Rate',
        'Segment Churn Rate (Peak Age Group: 46-60)',
        'High-Value Churn Ratio',
        'Geographic Risk Index (Germany vs Min)',
        'Engagement Drop Indicator (Inactive vs Active Churn Gap)'
    ],
    'Description': [
        '% customers who exited overall',
        'Highest risk churn % within demographic segments',
        'Churn % among premium / high-balance customers (>€100k)',
        'Ratio/Exposure multiplier of highest risk region (Germany)',
        'Extra % points of churn caused by account inactivity'
    ],
    'KPI Value': [
        f"{overall_churn_rate:.2f}%",
        f"{age_churn.max():.2f}% ({age_churn.idxmax()})",
        f"{high_val_churn_ratio:.2f}%",
        f"{geo_risk_index:.2f}x (Germany: {max_geo_churn:.1f}%)",
        f"+{engagement_drop_indicator:.2f}% higher churn for inactive"
    ]
})

display(kpi_summary)

OFFICIAL PROJECT KEY PERFORMANCE INDICATORS (KPIs) SUMMARY


,KPI Name,Description,KPI Value
0,Overall Churn Rate,% customers who exited overall,20.37%
1,Segment Churn Rate (Peak Age Group: 46-60),Highest risk churn % within demographic segments,51.12% (46–60)
2,High-Value Churn Ratio,Churn % among premium / high-balance customers...,0.00%
3,Geographic Risk Index (Germany vs Min),Ratio/Exposure multiplier of highest risk regi...,2.01x (Germany: 32.4%)
4,Engagement Drop Indicator (Inactive vs Active ...,Extra % points of churn caused by account inac...,+12.58% higher churn for inactive


In [26]:
# Save processed data to CSV for the Streamlit dashboard
df_clean.to_csv('European_Bank.csv', index=False)
print("Processed dataset saved as 'European_Bank.csv'!")

Processed dataset saved as 'European_Bank.csv'!


In [27]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np

# 1. Page Configuration & Styling
st.set_page_config(
    page_title="European Banking Churn Analytics",
    page_icon="🏦",
    layout="wide"
)

# Custom header layout
st.markdown("<h1 style='color: #1f77b4;'>🏦 Customer Segmentation & Churn Pattern Analytics</h1>", unsafe_allow_html=True)
st.markdown("**Unified Mentor | European Central Bank Decision Support Dashboard**")
st.markdown("---")

# 2. Load and Automatically Process 'European_Bank.csv'
@st.cache_data
def load_and_process_data():
    try:
        df = pd.read_csv('European_Bank.csv')
    except FileNotFoundError:
        st.error("'European_Bank.csv' not found! Please make sure your CSV file is uploaded in the same folder.")
        return pd.DataFrame()

    # Clean unnecessary unique identifier columns if present
    if 'RowNumber' in df.columns:
        df = df.drop(columns=['RowNumber'])
    if 'Surname' in df.columns:
        df = df.drop(columns=['Surname'])

    # Feature Engineering / Required Segmentations
    df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 29, 45, 60, np.inf], labels=['<30', '30–45', '46–60', '60+'])
    df['CreditScoreBand'] = pd.cut(df['CreditScore'], bins=[0, 599, 750, np.inf], labels=['Low', 'Medium', 'High'])
    df['TenureGroup'] = pd.cut(df['Tenure'], bins=[-1, 2, 7, np.inf], labels=['New', 'Mid-term', 'Long-term'])
    df['BalanceSegment'] = pd.cut(df['Balance'], bins=[-1, 0, 100000, np.inf], labels=['Zero-balance', 'Low-balance', 'High-balance'])
    df['ChurnStatus'] = df['Exited'].map({0: 'Retained', 1: 'Churned'})
    return df

df = load_and_process_data()

if not df.empty:
    # 3. Sidebar User Capabilities (Segment Filters)
    st.sidebar.header("🔍 Interactive Segment Filters")
    selected_geo = st.sidebar.multiselect("Geography", options=df['Geography'].unique(), default=df['Geography'].unique())
    selected_gender = st.sidebar.multiselect("Gender", options=df['Gender'].unique(), default=df['Gender'].unique())
    selected_age = st.sidebar.multiselect("Age Group", options=df['AgeGroup'].dropna().unique(), default=df['AgeGroup'].dropna().unique())
    selected_balance = st.sidebar.multiselect("Balance Segment", options=df['BalanceSegment'].dropna().unique(), default=df['BalanceSegment'].dropna().unique())

    # Dynamic Filtering Application
    filtered_df = df[
        (df['Geography'].isin(selected_geo)) &
        (df['Gender'].isin(selected_gender)) &
        (df['AgeGroup'].isin(selected_age)) &
        (df['BalanceSegment'].isin(selected_balance))
    ]

    # 4. Dynamic KPI Updates Module
    st.subheader("📊 Dynamic Key Performance Indicators (KPIs)")
    kpi1, kpi2, kpi3, kpi4, kpi5 = st.columns(5)

    total_cust = len(filtered_df)
    churned_cust = filtered_df['Exited'].sum()
    churn_rate = (churned_cust / total_cust * 100) if total_cust > 0 else 0
    high_val_lost = filtered_df[(filtered_df['Exited'] == 1) & (filtered_df['Balance'] > 100000)]['Balance'].sum()
    active_ratio = (filtered_df['IsActiveMember'].mean() * 100) if total_cust > 0 else 0

    kpi1.metric("Total Customers", f"{total_cust:,}")
    kpi2.metric("Total Churned", f"{churned_cust:,}")
    kpi3.metric("Overall Churn Rate", f"{churn_rate:.2f}%")
    kpi4.metric("High-Value Capital Lost", f"€{high_val_lost/1e6:.2f}M")
    kpi5.metric("Active Member %", f"{active_ratio:.1f}%")

    st.markdown("---")

    # Consistent color palette for charts
    color_map = {'Retained': '#2b5c8f', 'Churned': '#d9534f'}

    # 5. Core Modules Organized via Professional Tabs
    tab1, tab2, tab3, tab4 = st.tabs([
        "📌 Overall Churn Summary",
        "🌍 Geography-wise Churn",
        "📈 Age & Tenure Comparison",
        "💎 High-Value Churn Explorer"
    ])

    with tab1:
        st.subheader("Overall Churn Summary & Product Distribution")
        c1, c2 = st.columns(2)
        with c1:
            fig_overall = px.pie(filtered_df, names="ChurnStatus", title="Proportion of Churned vs Retained Customers",
                                 color="ChurnStatus", color_discrete_map=color_map, hole=0.4)
            st.plotly_chart(fig_overall, use_container_width=True)
        with c2:
            fig_prod = px.histogram(filtered_df, x="NumOfProducts", color="ChurnStatus", barmode="group",
                                   title="Churn Distribution by Number of Products Held", color_discrete_map=color_map)
            st.plotly_chart(fig_prod, use_container_width=True)

    with tab2:
        st.subheader("Geography-wise Churn Visualization & Risk Index")
        c3, c4 = st.columns(2)
        with c3:
            fig_geo = px.histogram(filtered_df, x="Geography", color="ChurnStatus", barmode="group",
                                  title="Regional Churn Breakdown (France, Spain, Germany)", color_discrete_map=color_map)
            st.plotly_chart(fig_geo, use_container_width=True)
        with c4:
            geo_rate = filtered_df.groupby('Geography', observed=False)['Exited'].mean().reset_index()
            geo_rate['Churn Rate (%)'] = geo_rate['Exited'] * 100
            fig_geo_rate = px.bar(geo_rate, x="Geography", y="Churn Rate (%)", color="Geography",
                                  title="Geographic Risk Index (Churn Rate %)", text_auto='.1f')
            st.plotly_chart(fig_geo_rate, use_container_width=True)

    with tab3:
        st.subheader("Age & Tenure Churn Comparison")
        c5, c6 = st.columns(2)
        with c5:
            fig_age = px.histogram(filtered_df, x="AgeGroup", color="ChurnStatus", barmode="group",
                                  title="Churn Comparison Across Age Segments", color_discrete_map=color_map)
            st.plotly_chart(fig_age, use_container_width=True)
        with c6:
            fig_tenure = px.histogram(filtered_df, x="TenureGroup", color="ChurnStatus", barmode="group",
                                     title="Churn Comparison Across Tenure Groups", color_discrete_map=color_map)
            st.plotly_chart(fig_tenure, use_container_width=True)

    with tab4:
        st.subheader("High-Value Customer Churn Explorer")
        c7, c8 = st.columns(2)
        with c7:
            bal_rate = filtered_df.groupby('BalanceSegment', observed=False)['Exited'].mean().reset_index()
            bal_rate['Churn Rate (%)'] = bal_rate['Exited'] * 100
            fig_bal = px.bar(bal_rate, x="BalanceSegment", y="Churn Rate (%)", color="BalanceSegment",
                             title="Churn Rate by Balance Segment", text_auto='.1f')
            st.plotly_chart(fig_bal, use_container_width=True)
        with c8:
            fig_scatter = px.scatter(filtered_df, x="EstimatedSalary", y="Balance", color="ChurnStatus",
                                    opacity=0.6, title="Estimated Salary vs Balance (High-Value Risk)", color_discrete_map=color_map)
            st.plotly_chart(fig_scatter, use_container_width=True)

    # 6. Drill-Down Views Capability
    st.markdown("---")
    with st.expander("🔍 Click to Expand: Detailed Customer Drill-Down View (Raw Filtered Dataset)"):
        st.markdown("Examine individual records instantly based on your active sidebar filter selections:")
        st.dataframe(filtered_df[['CustomerId', 'Geography', 'Gender', 'Age', 'CreditScore', 'Balance', 'NumOfProducts', 'IsActiveMember', 'EstimatedSalary', 'ChurnStatus']], use_container_width=True)

Writing app.py
